# 🛰️ Radar Micro-Doppler AI
## Notebook 1: Exploratory Data Analysis & Signal Analysis

This notebook explores the helicopter micro-Doppler dataset:
- Dataset statistics and class distributions
- Raw IQ signal visualization
- STFT spectrogram visualization
- Physical parameter distributions (RPM, blade radius, tip velocity, SNR)

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import stft

# Project imports
from Dataset.loader import load_dataset, get_iq_matrix, get_labels
from Preprocessing.spectrogram import compute_spectrogram_single
from Preprocessing.features import extract_features, FEATURE_NAMES

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#e6edf3',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#e6edf3',
    'grid.color': '#30363d',
    'font.family': 'DejaVu Sans',
})
COLORS = ['#58a6ff', '#3fb950', '#f78166']
CLASS_NAMES = {2: '2-blade (UH-1)', 3: '3-blade (Gazelle)', 4: '4-blade (Apache/UH-60)'}
print("✅ Setup complete")

## 1. Load Dataset

In [ ]:
# Load a manageable subset for EDA
df = load_dataset('../helicopter_microdoppler_dataset.csv', nrows=3000)
print(f"Shape: {df.shape}")
df.head(3)

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Dataset Overview', fontsize=16, fontweight='bold', y=1.02)

# Class counts
counts = df['num_blades'].value_counts().sort_index()
bars = axes[0].bar([CLASS_NAMES[k] for k in counts.index], counts.values,
                   color=COLORS, edgecolor='white', linewidth=0.5, alpha=0.9)
axes[0].set_title('Class Distribution', fontsize=13)
axes[0].set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# SNR distribution per class
for i, (nb, name) in enumerate(CLASS_NAMES.items()):
    subset = df[df['num_blades'] == nb]['snr_db']
    axes[1].hist(subset, bins=30, alpha=0.7, color=COLORS[i], label=name, density=True)
axes[1].set_title('SNR Distribution by Class', fontsize=13)
axes[1].set_xlabel('SNR (dB)')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("Plot saved.")

## 3. Physical Parameters Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Physical Parameter Distributions by Helicopter Type', fontsize=15, fontweight='bold')

params = [('rpm', 'Rotor RPM'), ('radius_m', 'Blade Radius (m)'),
          ('tip_velocity_m_s', 'Tip Velocity (m/s)'), ('snr_db', 'SNR (dB)')]

for ax, (col, label) in zip(axes.flat, params):
    for i, (nb, name) in enumerate(CLASS_NAMES.items()):
        data = df[df['num_blades'] == nb][col]
        ax.hist(data, bins=40, alpha=0.65, color=COLORS[i], label=name, density=True)
    ax.set_title(label, fontsize=12)
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('parameter_distributions.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 4. Raw IQ Signal Visualization

In [ ]:
X_iq = get_iq_matrix(df)
y    = get_labels(df)
t    = np.linspace(0, 0.5, 500, endpoint=False)

fig, axes = plt.subplots(3, 2, figsize=(16, 10))
fig.suptitle('Raw IQ Signals — One Sample per Class', fontsize=15, fontweight='bold')

for row_idx, (n_blades, name) in enumerate(CLASS_NAMES.items()):
    idx = np.where(y == n_blades)[0][0]
    sig = X_iq[idx]

    axes[row_idx, 0].plot(t, sig.real, color=COLORS[row_idx], linewidth=0.8, alpha=0.9)
    axes[row_idx, 0].set_title(f'{name} — In-Phase (I)', fontsize=11)
    axes[row_idx, 0].set_xlabel('Time (s)')
    axes[row_idx, 0].set_ylabel('Amplitude')
    axes[row_idx, 0].grid(True, alpha=0.2)

    axes[row_idx, 1].plot(t, sig.imag, color=COLORS[row_idx], linewidth=0.8, alpha=0.9)
    axes[row_idx, 1].set_title(f'{name} — Quadrature (Q)', fontsize=11)
    axes[row_idx, 1].set_xlabel('Time (s)')
    axes[row_idx, 1].set_ylabel('Amplitude')
    axes[row_idx, 1].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('iq_signals.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 5. Micro-Doppler Spectrograms

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Micro-Doppler STFT Spectrograms by Helicopter Type', fontsize=15, fontweight='bold')

for col_idx, (n_blades, name) in enumerate(CLASS_NAMES.items()):
    idx = np.where(y == n_blades)[0][0]
    freqs, times, spec = compute_spectrogram_single(X_iq[idx], nperseg=64, noverlap=56)

    im = axes[col_idx].pcolormesh(times, freqs, spec, shading='gouraud', cmap='inferno')
    axes[col_idx].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes[col_idx].set_xlabel('Time (s)')
    axes[col_idx].set_ylabel('Frequency (Hz)')
    plt.colorbar(im, ax=axes[col_idx], label='Power (dB)')

    # Annotate blade flash rate
    bfr = n_blades * df[df['num_blades'] == n_blades]['rpm'].mean() / 60
    axes[col_idx].set_title(f'{name}\nBlade Flash Rate ≈ {bfr:.1f} Hz', fontsize=11)

plt.tight_layout()
plt.savefig('spectrograms.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("📊 Spectrograms reveal the distinct periodic structure per helicopter type.")

## 6. Feature Correlation Heatmap

In [ ]:
X_feat = extract_features(X_iq[:500])

import matplotlib.cm as cm

corr = np.corrcoef(X_feat.T)
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(FEATURE_NAMES)))
ax.set_yticks(range(len(FEATURE_NAMES)))
ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(FEATURE_NAMES, fontsize=9)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='Pearson Correlation')

# Annotate high correlations
for i in range(len(FEATURE_NAMES)):
    for j in range(len(FEATURE_NAMES)):
        if abs(corr[i,j]) > 0.7 and i != j:
            ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center',
                    fontsize=7, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"✅ EDA complete. Key insight: blade_flash_rate_hz is the most discriminative feature.")